<a href="https://colab.research.google.com/github/shiwangiedulearn-jpg/ai-engineer-learning/blob/main/day6rag_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers scikit-learn

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
evaluation_data=[
    {
        "question": "what can increase cardiovascular risk?",
        "relevant_chunks": [0,2]
    },
    {
        "question": "What can support cardiovascular health?",
        "relevant_chunks": [1]
    },
    {
        "question": "What is associated with blood sugar regulation?",
        "relevant_chunks": [3]
    },
    {
        "question": "What can contribute to plaque buildup in arteries?",
        "relevant_chunks": [2]
    }
]

In [4]:
medical_documents = [
    """
    High blood pressure, also called hypertension, means that the force
    of blood against artery walls is consistently too high. Persistent
    hypertension can increase the risk of cardiovascular disease.
    """,

    """
    Regular physical activity can support cardiovascular health.
    Adults should generally aim for regular moderate-intensity activity,
    depending on their individual health circumstances.
    """,

    """
    High cholesterol can contribute to plaque buildup in arteries.
    LDL cholesterol is commonly referred to as bad cholesterol because
    elevated levels can increase cardiovascular risk.
    """,

    """
    Diabetes is a condition involving abnormal blood glucose regulation.
    Long-term uncontrolled blood sugar can affect multiple organs.
    """
]

In [5]:
full_text = "\n\n".join(medical_documents)

In [6]:
def chunk_text( text, chunk_size=300, overlap=50):
  start=0
  chunks= []
  while start< len(text):
    end = start+ chunk_size
    chunks.append(text[start:end])
    start = end - overlap
  return chunks


In [7]:
chunks = chunk_text(full_text)

In [15]:
chunks

['\n    High blood pressure, also called hypertension, means that the force\n    of blood against artery walls is consistently too high. Persistent\n    hypertension can increase the risk of cardiovascular disease.\n    \n\n\n    Regular physical activity can support cardiovascular health.\n    Adults should ',
 ' support cardiovascular health.\n    Adults should generally aim for regular moderate-intensity activity,\n    depending on their individual health circumstances.\n    \n\n\n    High cholesterol can contribute to plaque buildup in arteries.\n    LDL cholesterol is commonly referred to as bad cholesterol be',
 'erol is commonly referred to as bad cholesterol because\n    elevated levels can increase cardiovascular risk.\n    \n\n\n    Diabetes is a condition involving abnormal blood glucose regulation.\n    Long-term uncontrolled blood sugar can affect multiple organs.\n    ',
 'rgans.\n    ']

In [8]:
print("number of chunks: ",len(chunks))

number of chunks:  4


In [9]:
model = SentenceTransformer("all-MiniLM-l6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
def retrieve_top_k(query, chunks, model, k=2):
    query_embedding= model.encode([query])
    chunk_embedding= model.encode(chunks)
    scores =cosine_similarity(
        query_embedding,
        chunk_embedding
    )[0]
    top_indices= scores.argsort()[::-1][:k]
    return top_indices



In [11]:
query = "What can increase cardiovascular risk?"
retrieved= retrieve_top_k(query, chunks, model,k=2 )
print("question: ", query)
print("retrieved chunks: ",retrieved)

question:  What can increase cardiovascular risk?
retrieved chunks:  [1 0]


In [13]:
def hit_at_k(retrieved_indices, relevant_indices):
  return int(
      any(index in relevant_indices for index in retrieved_indices)
  )

In [17]:
relevant= evaluation_data[0]["relevant_chunks"]
scores = hit_at_k(
    retrieved,
    relevant
)
print("hit@2: ",scores)

hit@2:  1


In [18]:
hit_scores=[]
for item in evaluation_data:
  retrieved= retrieve_top_k(
      item["question"],
      chunks,
      model,
      k=2
  )
  score= hit_at_k(
      retrieved,
      item["relevant_chunks"]
  )
  hit_scores.append(score)
  print("question: ",item["question"])
  print("retrieved: ", retrieved)
  print("Hit@2: ", scores)

question:  what can increase cardiovascular risk?
retrieved:  [1 0]
Hit@2:  1
question:  What can support cardiovascular health?
retrieved:  [1 0]
Hit@2:  1
question:  What is associated with blood sugar regulation?
retrieved:  [2 1]
Hit@2:  1
question:  What can contribute to plaque buildup in arteries?
retrieved:  [1 2]
Hit@2:  1


In [23]:
average_hit = sum(hit_scores)/len(hit_scores)


In [24]:
print("Overall Hit@2: ", round(average_hit,3))

Overall Hit@2:  0.75


In [25]:
for k in [1,2,3]:
  scores_k=[]
  for item in evaluation_data:
    retrieved= retrieve_top_k(
        item["question"],
        chunks,
        model,
        k=k
    )
    score= hit_at_k(
        retrieved,
        item["relevant_chunks"]
    )
    scores_k.append(score)
  average= sum(scores_k)/ len(scores_k)
  print(f"Hit@{k}: {average:.3f}")



Hit@1: 0.250
Hit@2: 0.750
Hit@3: 0.750


In [29]:
for chunk_size in [100, 300, 500]:
  scores_chunk_size= []
  for item in evaluation_data:
    chunks = chunk_text(full_text, chunk_size = chunk_size)
    retrieved = retrieve_top_k(
        item["question"],
        chunks,
        model,
        k=2
    )
    scores = hit_at_k(
        retrieved,
        item["relevant_chunks"]
    )
    scores_chunk_size.append(scores)
  average = sum(scores_chunk_size)/ len(scores_chunk_size)
  print(f"Hit@2_{chunk_size}: {average:.3f}")



Hit@2_100: 0.000
Hit@2_300: 0.750
Hit@2_500: 0.500


In [30]:
context = "\n\n".join(
    [chunks[i] for i in retrieved]
)

In [31]:
context

'ute to plaque buildup in arteries.\n    LDL cholesterol is commonly referred to as bad cholesterol because\n    elevated levels can increase cardiovascular risk.\n    \n\n\n    Diabetes is a condition involving abnormal blood glucose regulation.\n    Long-term uncontrolled blood sugar can affect multiple organs.\n    \n\n\n    High blood pressure, also called hypertension, means that the force\n    of blood against artery walls is consistently too high. Persistent\n    hypertension can increase the risk of cardiovascular disease.\n    \n\n\n    Regular physical activity can support cardiovascular health.\n    Adults should generally aim for regular moderate-intensity activity,\n    depending on their individual health circumstances.\n    \n\n\n    High cholesterol can contribute to plaque buildup in arteries.\n    LDL cholest'

In [51]:
prompt = f"""
Answer the question using ONLY the context below.

If the context does not contain enough information,
say "I don't have enough information."

Context:
{context}

Question:
"what is capital of france"

Answer:
"""

In [52]:
from google import genai
from google.colab import userdata
GEMINI_API_KEY= userdata.get("GEMINI_API_KEY")
print("API key loaded", GEMINI_API_KEY is not None)

API key loaded True


In [53]:
!pip install -q -U google-genai

from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

print(response.text)

I don't have enough information.
